# 03 · Feature Engineering
Builds every mandatory descriptive feature used across the 20 Streamlit pages. All logic here matches
`utils/feature_engineering.py` exactly, so the notebook and the live app never disagree.

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, '..')
from utils.feature_engineering import (
    engineer_application_features, aggregate_bureau_features, aggregate_pos_cash_features
)

app_train = pd.read_csv('../data/application_train.csv')
app_train.shape


(307511, 122)

## Application-Level Features

In [2]:
app_fe = engineer_application_features(app_train)
app_fe[['AGE_YEARS', 'AGE_GROUP', 'EMPLOYMENT_YEARS', 'EMPLOYMENT_GROUP',
        'INCOME_GROUP', 'INCOME_PER_FAMILY_MEMBER', 'CREDIT_TO_INCOME',
        'ANNUITY_TO_INCOME', 'GOODS_TO_INCOME', 'CREDIT_TO_GOODS']].head()


,AGE_YEARS,AGE_GROUP,EMPLOYMENT_YEARS,EMPLOYMENT_GROUP,INCOME_GROUP,INCOME_PER_FAMILY_MEMBER,CREDIT_TO_INCOME,ANNUITY_TO_INCOME,GOODS_TO_INCOME,CREDIT_TO_GOODS
0,25.9,20-30,1.7,1-3 Years,High,202500.0,2.007889,0.121978,1.733333,1.158397
1,45.9,41-50,3.3,3-5 Years,Very High,135000.0,4.790750,0.132217,4.183333,1.145199
2,52.1,51-60,0.6,<1 Year,Very Low,67500.0,2.000000,0.100000,2.000000,1.000000
3,52.0,51-60,8.3,5-10 Years,Low,67500.0,2.316167,0.219900,2.200000,1.052803
4,54.6,51-60,8.3,5-10 Years,Low,121500.0,4.222222,0.179963,4.222222,1.000000


In [3]:
app_fe['AGE_GROUP'].value_counts()


AGE_GROUP
31-40    82413
41-50    76479
51-60    68080
20-30    45530
60+      35009
Name: count, dtype: int64

In [4]:
app_fe['EMPLOYMENT_GROUP'].value_counts()


EMPLOYMENT_GROUP
5-10 Years              65436
1-3 Years               61704
Unemployed / Special    55374
3-5 Years               47271
10-20 Years             38719
<1 Year                 26366
20+ Years               12641
Name: count, dtype: int64

In [5]:
app_fe['INCOME_GROUP'].value_counts()


INCOME_GROUP
Low          85756
High         75513
Very Low     63671
Very High    47118
Middle       35453
Name: count, dtype: int64

In [6]:
app_fe[['CREDIT_TO_INCOME', 'ANNUITY_TO_INCOME', 'GOODS_TO_INCOME', 'CREDIT_TO_GOODS']].describe()


,CREDIT_TO_INCOME,ANNUITY_TO_INCOME,GOODS_TO_INCOME,CREDIT_TO_GOODS
count,307511.000000,307499.000000,307233.000000,307233.000000
mean,3.957570,0.180930,3.544322,1.122995
std,2.689728,0.094574,2.427708,0.124045
min,0.004808,0.000224,0.003885,0.150000
25%,2.018667,0.114782,1.840000,1.000000
50%,3.265067,0.162833,2.941176,1.118800
75%,5.159880,0.229067,4.615385,1.198000
max,84.736842,1.875965,84.736842,6.000000


## Bureau-Level Aggregates (per SK_ID_CURR)

In [7]:
bureau = pd.read_csv('../data/bureau.csv')
bureau_agg = aggregate_bureau_features(bureau)
bureau_agg.head()


,SK_ID_CURR,BUREAU_ACCOUNT_COUNT,ACTIVE_BUREAU_COUNT,CLOSED_BUREAU_COUNT,TOTAL_BUREAU_CREDIT,TOTAL_BUREAU_DEBT,AVG_BUREAU_CREDIT,MAX_BUREAU_OVERDUE,TOTAL_BUREAU_OVERDUE
0,100001,7,3,4,1453365.000,596686.5,207623.571429,0.0,0.0
1,100002,8,2,6,865055.565,245781.0,108131.945625,0.0,0.0
2,100003,4,1,3,1017400.500,0.0,254350.125000,0.0,0.0
3,100004,2,0,2,189037.800,0.0,94518.900000,0.0,0.0
4,100005,3,2,1,657126.000,568408.5,219042.000000,0.0,0.0


## POS/CASH-Level Aggregates (per SK_ID_CURR)

In [8]:
pos_cash = pd.read_csv('../data/POS_CASH_balance.csv')
pos_agg = aggregate_pos_cash_features(pos_cash)
pos_agg.head()


,SK_ID_CURR,POS_RECORD_COUNT,AVG_DPD,MAX_DPD,TOTAL_DPD_EVENTS,AVG_INSTALMENTS_REMAINING,COMPLETED_CONTRACTS
0,1,1,NaN,NaN,0,NaN,0
1,100003,2,0.0,0.0,0,1.00,0
2,100005,1,0.0,0.0,0,9.00,0
3,100007,4,0.0,0.0,0,13.25,0
4,100008,1,0.0,0.0,0,28.00,0


## Merge Into a Single Customer-Level Feature Table

In [9]:
customer_features = (
    app_fe
    .merge(bureau_agg, on='SK_ID_CURR', how='left')
    .merge(pos_agg, on='SK_ID_CURR', how='left')
)
customer_features['TOTAL_BUREAU_DEBT'] = customer_features['TOTAL_BUREAU_DEBT'].fillna(0)
customer_features.shape


(307511, 147)

In [10]:
customer_features.to_csv('../data/customer_features_engineered.csv', index=False)
print('Saved engineered feature table.')


Saved engineered feature table.
